# 01 — Khám phá và tiền xử lý dữ liệu Parkinson

**Mục tiêu:** lập hồ sơ chất lượng dữ liệu và ghi nhận kết quả của checklist tiền xử lý. Notebook được tạo ở trạng thái chưa chạy.

> Đặt `D07 - parkinsons.data.xlsx` vào thư mục `data/` trước khi thực thi. Không chỉnh sửa file gốc. Checklist 14 mục bên dưới là khung kỹ thuật đề xuất; hãy đối chiếu tên bước với tài liệu chính thức của giảng viên.


In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import quality_profile, iqr_outlier_report, mad_outlier_report, find_target_column, find_identifier_columns

DATA_DIR = PROJECT_ROOT / "data"
RAW_PATH = DATA_DIR / "D07 - parkinsons.data.xlsx"
PROCESSED_PATH = DATA_DIR / "parkinsons_processed.csv"
RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)


## 1. Đọc dữ liệu gốc

Nếu file của nhóm có tên khác hoặc định dạng CSV, cập nhật `RAW_PATH` và cách đọc tương ứng. Cell này không tự tạo dữ liệu giả.


In [ ]:
if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy dữ liệu: {RAW_PATH}\n"
        "Hãy đặt file Excel gốc vào thư mục data/ theo README.md."
    )

if RAW_PATH.suffix.lower() in {".xlsx", ".xls"}:
    df_raw = pd.read_excel(RAW_PATH)
elif RAW_PATH.suffix.lower() == ".csv":
    df_raw = pd.read_csv(RAW_PATH)
else:
    raise ValueError(f"Định dạng chưa hỗ trợ: {RAW_PATH.suffix}")

df = df_raw.copy()
df.columns = [str(c).strip() for c in df.columns]
print(f"Số dòng: {df.shape[0]:,} | Số cột: {df.shape[1]:,}")
display(df.head())


## 2. Checklist tiền xử lý — 14 mục

| STT | Hạng mục | Nội dung kiểm tra |
|---:|---|---|
| 1 | Bảo toàn dữ liệu gốc | Chỉ đọc `df_raw`; mọi chỉnh sửa thực hiện trên bản sao |
| 2 | Kích thước dữ liệu | Số dòng, số cột |
| 3 | Cấu trúc mẫu | Xem vài dòng đầu/cuối và tên cột |
| 4 | Kiểu dữ liệu | Kiểm tra kiểu mỗi cột, chuyển kiểu chỉ khi có căn cứ |
| 5 | Giá trị thiếu | Đếm số lượng và tỷ lệ thiếu |
| 6 | Bản ghi trùng | Đếm bản ghi trùng hoàn toàn |
| 7 | Số giá trị phân biệt | Phát hiện cột gần như hằng số/định danh |
| 8 | Giá trị bất thường theo miền | Kiểm tra min, max, giá trị âm hoặc miền không hợp lý theo ý nghĩa cột |
| 9 | Phân bố nhãn | Xác định mất cân bằng lớp và mã hóa nhãn |
| 10 | Cột định danh | Nhận diện `name`/ID để không dùng trực tiếp làm đặc trưng |
| 11 | Thống kê mô tả | Trung bình, độ lệch chuẩn, tứ phân vị và phân phối |
| 12 | Ngoại lai IQR | Dùng ngưỡng Q1 − 1.5×IQR và Q3 + 1.5×IQR |
| 13 | Ngoại lai MAD | Dùng robust z-score; lưu ý trường hợp MAD = 0 |
| 14 | Chuẩn hóa và lưu bản làm việc | Minh họa StandardScaler/MinMaxScaler; không fit scaler trên toàn bộ dữ liệu khi đánh giá mô hình |

Không tự động xóa ngoại lai chỉ vì một điểm bị gắn cờ. Hãy xem xét ý nghĩa dữ liệu và ghi lại quyết định.


## 3. Lập hồ sơ chất lượng dữ liệu (mục 2–7)


In [ ]:
print("Kiểu dữ liệu:")
display(df.dtypes.rename("dtype").to_frame())

print("Hồ sơ chất lượng:")
quality = quality_profile(df)
display(quality)

print("Số bản ghi trùng hoàn toàn:", int(df.duplicated().sum()))
print("Cột chỉ có một giá trị phân biệt:", df.columns[df.nunique(dropna=True) <= 1].tolist())
print("Thống kê mô tả các cột số:")
display(df.describe(include=[np.number]).T)


## 4. Kiểm tra cột mục tiêu và định danh (mục 9–10)

Bộ Parkinsons phổ biến dùng `status` (1 = Parkinson, 0 = không Parkinson). Hãy xác nhận ý nghĩa mã nhãn trong đúng bộ dữ liệu nhóm đang sử dụng.


In [ ]:
TARGET_COL = find_target_column(df.columns)
if TARGET_COL is None:
    raise ValueError("Không tự nhận diện được cột nhãn. Hãy đặt TARGET_COL đúng tên cột mục tiêu.")
ID_COLS = find_identifier_columns(df.columns)
print("Cột mục tiêu:", TARGET_COL)
print("Cột định danh dự kiến loại khỏi mô hình:", ID_COLS)
print("Phân bố nhãn (bao gồm NaN nếu có):")
display(df[TARGET_COL].value_counts(dropna=False).rename_axis(TARGET_COL).to_frame("count"))
ax = df[TARGET_COL].value_counts(dropna=False).plot(kind="bar", title="Phân bố nhãn", rot=0)
ax.set_xlabel(TARGET_COL)
ax.set_ylabel("Số mẫu")
plt.tight_layout()
plt.show()


## 5. Khảo sát ngoại lai bằng IQR và MAD (mục 11–13)

Hai bảng sau chỉ **báo cáo** điểm bị gắn cờ; không xóa hay thay thế dữ liệu tự động. Loại cột nhãn và định danh khỏi việc rà soát ngoại lai ở các đặc trưng.


In [ ]:
exclude_for_outliers = [TARGET_COL, *ID_COLS]
print("Báo cáo IQR:")
iqr_report = iqr_outlier_report(df, exclude_cols=exclude_for_outliers, multiplier=1.5)
display(iqr_report)

print("Báo cáo MAD (robust z-score threshold = 3.5):")
mad_report = mad_outlier_report(df, exclude_cols=exclude_for_outliers, threshold=3.5)
display(mad_report)


## 6. Trực quan hóa một số đặc trưng số

Các biểu đồ phục vụ khám phá dữ liệu; hãy chọn đặc trưng phù hợp để đưa vào báo cáo thay vì đưa tất cả biểu đồ.


In [ ]:
numeric_features = [
    c for c in df.select_dtypes(include=np.number).columns
    if c != TARGET_COL and c not in ID_COLS
]
if numeric_features:
    sample_features = numeric_features[:6]
    df[sample_features].hist(figsize=(14, 8), bins=20, edgecolor="black")
    plt.suptitle("Phân phối một số đặc trưng số")
    plt.tight_layout()
    plt.show()

    if len(sample_features) >= 2:
        plt.figure(figsize=(10, 7))
        sns.heatmap(df[sample_features].corr(), cmap="coolwarm", center=0, annot=False)
        plt.title("Tương quan giữa một số đặc trưng")
        plt.tight_layout()
        plt.show()


## 7. Minh họa Scaling / Normalization (mục 14)

Phần dưới chỉ minh họa khác biệt về thang đo trên các cột số. Trong notebook mô hình, scaler phải được fit trên tập train hoặc nằm trong pipeline; không dùng kết quả scaler fit từ toàn bộ dữ liệu để báo cáo hiệu năng test.


In [ ]:
if numeric_features:
    demo_cols = numeric_features[: min(5, len(numeric_features))]
    demo = df[demo_cols].apply(pd.to_numeric, errors="coerce")
    demo = demo.fillna(demo.median(numeric_only=True))
    standard_demo = pd.DataFrame(StandardScaler().fit_transform(demo), columns=demo_cols, index=demo.index)
    minmax_demo = pd.DataFrame(MinMaxScaler().fit_transform(demo), columns=demo_cols, index=demo.index)
    print("Ví dụ StandardScaler (z-score):")
    display(standard_demo.head())
    print("Ví dụ MinMaxScaler (thường đưa về [0, 1]):")
    display(minmax_demo.head())
else:
    print("Không có cột số để minh họa scaling.")


## 8. Tạo bản CSV làm việc (tùy chọn)

Cell này loại bản ghi trùng hoàn toàn và xử lý giá trị thiếu ở các cột đầu vào theo quy tắc đơn giản: cột số dùng median, cột không phải số dùng mode. Bản CSV vẫn giữ cột định danh để đối chiếu; notebook 02 sẽ loại định danh khỏi `X`. Không tự động loại ngoại lai.

**Trước khi lưu:** xem báo cáo thiếu dữ liệu và xác nhận cách điền khuyết có phù hợp với yêu cầu của giảng viên không.


In [ ]:
df_clean = df.drop_duplicates().copy()
if TARGET_COL in df_clean.columns:
    # Không tự suy đoán nhãn bị thiếu; bỏ các dòng không có nhãn.
    df_clean = df_clean.dropna(subset=[TARGET_COL])

for col in df_clean.columns:
    if col == TARGET_COL or df_clean[col].isna().sum() == 0:
        continue
    if pd.api.types.is_numeric_dtype(df_clean[col]):
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())
    else:
        modes = df_clean[col].mode(dropna=True)
        if not modes.empty:
            df_clean[col] = df_clean[col].fillna(modes.iloc[0])

print("Kích thước trước làm sạch:", df.shape)
print("Kích thước bản làm việc:", df_clean.shape)
print("Giá trị thiếu còn lại:", int(df_clean.isna().sum().sum()))
# Bỏ comment dòng dưới sau khi đã xem xét và chấp nhận quy tắc làm sạch.
# df_clean.to_csv(PROCESSED_PATH, index=False, encoding="utf-8-sig")
# print(f"Đã lưu bản CSV làm việc tại: {PROCESSED_PATH}")


## 9. Tổng kết tiền xử lý — điền sau khi chạy

- Số mẫu ban đầu / sau xử lý:
- Cột mục tiêu và ý nghĩa nhãn:
- Tỷ lệ thiếu và cách xử lý:
- Số bản ghi trùng:
- Các đặc trưng bị gắn cờ ngoại lai theo IQR/MAD và quyết định xử lý:
- Những cột định danh hoặc cột cần loại khỏi mô hình:
- Lý do có/không dùng Scaling:
- Tệp CSV đã lưu (nếu có):
